# Competencia: ¿qué tiene esta hoja de café?

**Sesión 4 · IA Intermedio · EstudIA 2026**

La roya es la enfermedad que más daño le hace al café. Detectarla temprano permite tratar el cafetal
antes de perder la cosecha. Cada foto es una hoja de café robusta, tomada con un celular en la finca, y
tu modelo tiene que decir qué tiene:

| Etiqueta | Qué tiene |
|---|---|
| `sano` | Hoja sana |
| `acaro_rojo` | Daño de ácaro rojo: la hoja se pone amarillenta o rojiza y opaca |
| `roya_leve` | Roya en su primera etapa: pocas manchas pequeñas |
| `roya_avanzada` | Roya más avanzada: manchas grandes o muchas, con polvo anaranjado |

**La métrica: exactitud balanceada.** Hay muchas más hojas sanas que enfermas. Si tu modelo dijera
“sano” para todas, acertaría en más de la mitad de las fotos de prueba y no serviría para nada. Por eso
la tabla no usa la exactitud de siempre, sino la **exactitud balanceada**: calculamos qué fracción de
cada clase acertó el modelo (el *recall* de cada clase) y promediamos esos cuatro números. Cada clase
pesa lo mismo, tenga muchas fotos o pocas. Responder “sano” para todo da 25 %.

**Reglas**

1. Ajusta un **modelo preentrenado de Hugging Face** con el `Trainer`. Puedes cambiar el modelo, qué
   partes se entrenan, el aumento de datos, las épocas, la tasa de aprendizaje y todo lo demás. No uses
   modelos que ya fueron ajustados con fotos de hojas de café.
2. Tienes 320 fotos con etiqueta (80 por clase) y 1.111 sin etiqueta. Separa tu propia validación: no
   tienes las etiquetas de prueba.
3. Envía tus predicciones todas las veces que quieras (una vez cada 10 segundos). Gana la mayor
   **exactitud balanceada**.
4. La tabla en vivo usa **la mitad** de las fotos de prueba. Al final, el profesor revela el resultado
   con **la otra mitad**: ese decide la competencia.

**Cómo empezar.** *Archivo › Guardar una copia en Drive*. Si Colab te da GPU, actívala (*Entorno de
ejecución › Cambiar tipo de entorno de ejecución › GPU T4*): la necesitas para ajustar todo el modelo.
El notebook, tal como viene, también funciona sin GPU. Ejecútalo completo (*Entorno de ejecución ›
Ejecutar todas*). Cuando te pida la clave, cópiala de la página de la competencia en la plataforma y
pégala. Así haces tu primer envío. Después, mejora tu modelo.

In [ ]:
# Preparación: librerías y datos. Ejecútala una vez.
import os, time, zipfile, urllib.request
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s04")

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from transformers import AutoImageProcessor, AutoModelForImageClassification, Trainer, TrainingArguments

GPU = torch.cuda.is_available()

def descargar(nombre):
    """Baja un archivo de la carpeta pública del curso (solo la primera vez) y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

for nombre in ("cafe_entrenamiento.zip", "cafe_prueba.zip"):
    with zipfile.ZipFile(descargar(nombre)) as z:
        z.extractall("archivos_curso")

CLASES = ["sano", "acaro_rojo", "roya_leve", "roya_avanzada"]
rutas = sorted(os.path.join(r, f) for r, _, fs in os.walk("archivos_curso/cafe/entrenamiento") for f in fs)
fotos = np.stack([np.asarray(Image.open(r).convert("RGB")) for r in rutas])
etiquetas = np.array([CLASES.index(os.path.basename(os.path.dirname(r))) for r in rutas])
rutas_prueba = sorted(os.listdir("archivos_curso/cafe/prueba"))
ids_prueba = [int(r.split(".")[0]) for r in rutas_prueba]
fotos_prueba = np.stack([np.asarray(Image.open(f"archivos_curso/cafe/prueba/{r}").convert("RGB")) for r in rutas_prueba])
print(f"Listo: {miles(len(fotos))} fotos con etiqueta y {miles(len(fotos_prueba))} de prueba, de "
      f"{fotos.shape[1]} × {fotos.shape[2]} píxeles.",
      f"GPU: {torch.cuda.get_device_name(0)}." if GPU else "Sin GPU: el modelo tal como viene funciona igual, más despacio.")

In [ ]:
# @title Función para enviar tus predicciones (no hace falta cambiarla)
import getpass, json, urllib.error, urllib.request

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")

def enviar(ids, predicciones, nota=""):
    """Envía tus predicciones a la plataforma del curso y muestra tu exactitud balanceada en la tabla.

    ids: los ids de las fotos de prueba. predicciones: una etiqueta por foto, como "roya_leve".
    nota: un texto corto para recordar qué probaste (opcional)."""
    global _CLAVE
    if not _CLAVE:
        _CLAVE = getpass.getpass("Pega tu clave (plataforma › Competencia › Copiar) y oprime Enter: ").strip()
    cuerpo = json.dumps({"ids": [int(i) for i in ids], "predicciones": [str(p) for p in predicciones],
                         "nota": str(nota)[:80]}).encode()
    req = urllib.request.Request(f"{PLATAFORMA}/api/competencia/s04/envios", data=cuerpo, method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {_CLAVE}",
                                          "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""  # la próxima vez la pide de nuevo
        print("No se pudo enviar:", res.get("error", f"error {e.code}"))
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    metrica = res.get("metrica", "Exactitud")
    print(f"Envío {res['envio']} recibido. {metrica} en la tabla: {pct(res.get('puntaje', res['exactitud']))}.")
    print(f"Tu mejor resultado: {pct(res['mejor'])} · puesto {res['puesto']} de {res['participantes']} ({res['alias']}).")

In [ ]:
# @title Funciones para dibujar curvas y leer la historia del Trainer
def curvas(historia, titulo, metrica="exactitud"):
    """Pérdida de entrenamiento y de validación, y la métrica de validación, por época."""
    epocas = range(1, len(historia["val_loss"]) + 1)
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4))
    ejes[0].plot(epocas, historia["loss"], label="Entrenamiento", color="#000000", lw=2)
    ejes[0].plot(epocas, historia["val_loss"], label="Validación", color="#CB318E", lw=2)
    ejes[0].set_title("Pérdida"); ejes[0].legend(frameon=False, fontsize=9)
    ejes[1].plot(epocas, historia["val_metrica"], color="#CB318E", lw=2)
    ejes[1].set_title(f"{metrica.capitalize()} en validación"); ejes[1].set_ylim(0, 1)
    for eje in ejes:
        eje.set_xlabel("Época"); eje.grid(alpha=0.3); eje.spines[["top", "right"]].set_visible(False)
    fig.suptitle(titulo, fontweight="bold"); fig.tight_layout(); plt.show()

def historia_de(trainer):
    """Saca de trainer.state.log_history la pérdida por época y lo que midió en validación."""
    h = {"loss": [], "val_loss": [], "val_metrica": []}
    for fila in trainer.state.log_history:
        if "loss" in fila and "eval_loss" not in fila:
            h["loss"].append(round(float(fila["loss"]), 5))
        if "eval_loss" in fila:
            h["val_loss"].append(round(float(fila["eval_loss"]), 5))
            h["val_metrica"].append(round(float(fila["eval_metrica"]), 5))
    return h

---
## 1. Los datos

In [ ]:
fig, ejes = plt.subplots(len(CLASES), 6, figsize=(13, 9))
for c, clase in enumerate(CLASES):
    for j, i in enumerate(np.where(etiquetas == c)[0][:6]):
        ejes[c, j].imshow(fotos[i]); ejes[c, j].axis("off")
    ejes[c, 0].set_title(clase, fontsize=11, loc="left")
plt.tight_layout(); plt.show()

### 1.1 Entrenamiento y validación

Separamos el 20 % como validación, con el mismo número de fotos por clase: 64 para entrenar y 16 para
validar de cada clase. Con tan pocas fotos, la validación es ruidosa: dos o tres puntos de diferencia
pueden ser azar. Úsala para detectar cambios grandes, como pasar de congelar el cuerpo a ajustarlo todo.

In [ ]:
i_ent, i_val = train_test_split(np.arange(len(fotos)), test_size=0.2, stratify=etiquetas, random_state=42)
X_ent, y_ent, X_val, y_val = fotos[i_ent], etiquetas[i_ent], fotos[i_val], etiquetas[i_val]
print(f"Entrenamiento: {len(X_ent)} · Validación: {len(X_val)} · Prueba: {miles(len(fotos_prueba))}")

### 1.2 La métrica: exactitud balanceada

La exactitud balanceada es el promedio del *recall* de cada clase. Así la calculamos a mano, y así la
calcula `scikit-learn`. Con un modelo que responde siempre “sano”:

In [ ]:
def exactitud_balanceada(verdad, prediccion):
    """Promedio, entre las clases, de la fracción de cada clase que el modelo acertó."""
    verdad, prediccion = np.asarray(verdad), np.asarray(prediccion)
    return float(np.mean([(prediccion[verdad == c] == c).mean() for c in np.unique(verdad)]))

siempre_sano = np.zeros_like(y_val)
print("Exactitud:", pct((siempre_sano == y_val).mean()), "(en la prueba, con muchas más hojas sanas, sería más alta)")
print("Exactitud balanceada:", pct(exactitud_balanceada(y_val, siempre_sano)),
      "· con scikit-learn:", pct(balanced_accuracy_score(y_val, siempre_sano)))

---
## 2. Tu modelo

Las piezas que puedes cambiar:

- `MODELO_BASE`: el modelo preentrenado de Hugging Face.
- `CONGELAR`: si es `True`, el cuerpo no se entrena y solo aprende la cabeza.
- `AUMENTAR`: si es `True`, cada lote de entrenamiento pasa por `aumentar(x)`, que puedes escribir tú.
- `EPOCAS`, `TASA` y `LOTE`: las épocas, la tasa de aprendizaje y las fotos por lote.

Así como vienen funcionan, también sin GPU, pero dejan mucho por ganar. Ejecútalas tal cual para tu
primer envío y después mejóralas.

In [ ]:
MODELO_BASE = "facebook/convnext-tiny-224"
CONGELAR = True
AUMENTAR = False
EPOCAS = 5
TASA = 1e-3
LOTE = 16

def aumentar(x):
    """Recibe un lote de fotos (n, 3, 224, 224) con valores entre 0 y 1 y devuelve otro del mismo tamaño."""
    return x

In [ ]:
PROCESADOR = AutoImageProcessor.from_pretrained(MODELO_BASE)
MEDIA = torch.tensor(PROCESADOR.image_mean).view(1, 3, 1, 1)
DESV = torch.tensor(PROCESADOR.image_std).view(1, 3, 1, 1)

class Fotos(torch.utils.data.Dataset):
    def __init__(self, X, y=None):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        ejemplo = {"pixel_values": torch.from_numpy(self.X[i])}
        if self.y is not None:   # las fotos de prueba no tienen etiqueta
            ejemplo["labels"] = int(self.y[i])
        return ejemplo

def juntar(lote, entrenando=False):
    x = torch.stack([e["pixel_values"] for e in lote]).permute(0, 3, 1, 2).float() / 255
    if entrenando and AUMENTAR:
        x = aumentar(x)
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
    salida = {"pixel_values": (x - MEDIA) / DESV}
    if "labels" in lote[0]:
        salida["labels"] = torch.tensor([e["labels"] for e in lote])
    return salida

def metrica(prediccion):
    return {"metrica": balanced_accuracy_score(prediccion.label_ids, prediccion.predictions.argmax(-1))}

def crear_modelo():
    modelo = AutoModelForImageClassification.from_pretrained(
        MODELO_BASE, id2label=dict(enumerate(CLASES)),
        label2id={c: i for i, c in enumerate(CLASES)}, ignore_mismatched_sizes=True)
    if CONGELAR:   # todo lo que no sea la cabeza se queda quieto
        cabeza = modelo.classifier
        for p in modelo.parameters():
            p.requires_grad = False
        for p in cabeza.parameters():
            p.requires_grad = True
    return modelo

def entrenar(X, y, X_val=None, y_val=None):
    """Ajusta un modelo nuevo con X, y. Si le das validación, la mide al final de cada época."""
    torch.manual_seed(0)
    args = TrainingArguments(
        output_dir="ajustes", num_train_epochs=EPOCAS, learning_rate=TASA,
        per_device_train_batch_size=LOTE, per_device_eval_batch_size=64, warmup_steps=0.1,
        eval_strategy="epoch" if X_val is not None else "no", logging_strategy="epoch", save_strategy="no",
        fp16=GPU, report_to="none", seed=0, remove_unused_columns=False)
    trainer = Trainer(model=crear_modelo(), args=args, train_dataset=Fotos(X, y),
                      eval_dataset=Fotos(X_val, y_val) if X_val is not None else None,
                      data_collator=lambda lote: juntar(lote, entrenando=True), compute_metrics=metrica)
    inicio = time.time()
    trainer.train()
    print(f"{EPOCAS} épocas en {time.time() - inicio:.0f} s.")
    trainer.data_collator = juntar
    return trainer

def predecir(trainer, X):
    return trainer.predict(Fotos(X)).predictions.argmax(-1)

### 2.1 Entrenar y validar

In [ ]:
trainer = entrenar(X_ent, y_ent, X_val, y_val)
curvas(historia_de(trainer), MODELO_BASE + (" · cuerpo congelado" if CONGELAR else " · todo el modelo"), "exactitud balanceada")
pred_val = predecir(trainer, X_val)
print("Validación · exactitud balanceada:", pct(balanced_accuracy_score(y_val, pred_val)))
ConfusionMatrixDisplay(confusion_matrix(y_val, pred_val, labels=range(len(CLASES))), display_labels=CLASES).plot(
    cmap="Purples", colorbar=False); plt.xlabel("Lo que respondió"); plt.ylabel("La clase real"); plt.show()

### 2.2 Predecir la prueba y enviar

Cambia la nota para recordar qué probaste en cada envío. La primera vez te pide la clave.

In [ ]:
predicciones = [CLASES[i] for i in predecir(trainer, fotos_prueba)]
enviar(ids_prueba, predicciones, nota="primer envío")

---
## 3. Pistas para subir en la tabla

Prueba una cosa a la vez, mira la exactitud balanceada de **validación** y envía cuando mejore.

<details><summary>Pista 1</summary>

El modelo viene con el cuerpo congelado: solo aprende la cabeza. ¿Qué vimos en la demostración cuando
ajustamos todo el modelo? Si lo haces, baja la tasa de aprendizaje, por ejemplo a `1e-4`, y usa GPU.
</details>

<details><summary>Pista 2</summary>

Aumento de datos. Una hoja fotografiada de lado o volteada sigue teniendo la misma enfermedad. Pon
`AUMENTAR = True` y escribe `aumentar(x)`: `torch.flip` y `torch.rot90` sirven. ¿Qué cambios de color
o de brillo no le quitarían la etiqueta a una hoja?
</details>

<details><summary>Pista 3</summary>

Las épocas. Mira las curvas: ¿la pérdida de validación sigue bajando al final? Con más épocas o con
aumento de datos, el modelo puede seguir aprendiendo.
</details>

<details><summary>Pista 4</summary>

Otro modelo base. Algunos que funcionan con este notebook: `google/vit-base-patch16-224`,
`microsoft/swin-tiny-patch4-window7-224`, `facebook/convnext-small-224` y `facebook/dinov2-small`. Los
más grandes necesitan GPU. Cuando cambies el modelo, vuelve a probar la tasa de aprendizaje.
</details>

<details><summary>Pista 5</summary>

Cuando ya elegiste el modelo, las épocas y la tasa, entrena el modelo final con las 320 fotos
(entrenamiento más validación) antes de predecir la prueba: `trainer = entrenar(fotos, etiquetas)`.
</details>

**Una advertencia.** Si eliges tus cambios mirando solo la tabla en vivo, terminas ajustando el modelo
a esa mitad de la prueba. El resultado final se calcula con la otra mitad.

**Fuente de los datos:** Parraga-Alava, J., Cusme, K., Loor, A. y Santander, E. (2019). RoCoLe: A robusta coffee
leaf images dataset for evaluation of machine learning based methods in plant diseases recognition. *Data in
Brief*, 25. Licencia CC BY 4.0. Copia en Hugging Face: `Project-AgML/RoCoLe_disease_detection`. Juntamos los
niveles 2, 3 y 4 de roya en `roya_avanzada`, quitamos las fotos repetidas, recortamos cada foto al cuadrado
del centro y la redujimos a 224 × 224 píxeles.